# 03 — Baseline MLP Neuronska Mreža
**Cilj:** Izgraditi i istrenirati osnovnu MLP mrežu (Multi-Layer Perceptron) kao baseline model.  
Poredimo tri pristupa za tretman nebalansiranosti: `class_weight`, `SMOTE`, `SMOTE+Under`.

---

## 0. Imports i konfiguracija

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
import warnings

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve, f1_score
)

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

PROC = '../data/processed/'
MODELS = '../models/'
RESULTS = '../results/'
os.makedirs(MODELS, exist_ok=True)

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

## 1. Učitavanje procesiranih podataka

In [ ]:
X_train        = np.load(f'{PROC}X_train.npy')
X_val          = np.load(f'{PROC}X_val.npy')
X_test         = np.load(f'{PROC}X_test.npy')
y_train        = np.load(f'{PROC}y_train.npy')
y_val          = np.load(f'{PROC}y_val.npy')
y_test         = np.load(f'{PROC}y_test.npy')
X_train_smote  = np.load(f'{PROC}X_train_smote.npy')
y_train_smote  = np.load(f'{PROC}y_train_smote.npy')
X_train_comb   = np.load(f'{PROC}X_train_combined.npy')
y_train_comb   = np.load(f'{PROC}y_train_combined.npy')

with open(f'{PROC}class_weights.json') as f:
    cw = json.load(f)
class_weight = {int(k): v for k, v in cw.items()}

INPUT_DIM = X_train.shape[1]
print(f'Input dim: {INPUT_DIM}')
print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')

## 2. Helper funkcije

In [ ]:
def plot_history(history, title='Training History', save_path=None):
    """Prikazuje loss i metrics kroz epohe."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    axes[0].plot(history.history['loss'], label='Train Loss')
    axes[0].plot(history.history['val_loss'], label='Val Loss')
    axes[0].set_title(f'{title} — Loss')
    axes[0].set_xlabel('Epoha')
    axes[0].legend()
    
    axes[1].plot(history.history['auc'], label='Train AUC-PR')
    axes[1].plot(history.history['val_auc'], label='Val AUC-PR')
    axes[1].set_title(f'{title} — AUC-PR')
    axes[1].set_xlabel('Epoha')
    axes[1].legend()
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
    plt.show()


def evaluate_model(model, X_test, y_test, threshold=0.5, title='Model', save_path=None):
    """Kompletna evaluacija: CM, ROC, PR kriva, klasifikacioni izveštaj."""
    y_prob = model.predict(X_test, verbose=0).ravel()
    y_pred = (y_prob >= threshold).astype(int)
    
    roc_auc = roc_auc_score(y_test, y_prob)
    pr_auc  = average_precision_score(y_test, y_prob)
    f1      = f1_score(y_test, y_pred)
    
    print(f'\n=== {title} ===')
    print(f'ROC-AUC:  {roc_auc:.4f}')
    print(f'PR-AUC:   {pr_auc:.4f}  ← glavna metrika za imbalanced')
    print(f'F1-Score: {f1:.4f}')
    print()
    print(classification_report(y_test, y_pred, target_names=['Legitimna', 'Fraud']))
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'])
    axes[0].set_title('Confusion Matrix')
    axes[0].set_ylabel('Stvarno')
    axes[0].set_xlabel('Predviđeno')
    
    # ROC kriva
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.4f}')
    axes[1].plot([0, 1], [0, 1], 'k--')
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title('ROC Kriva')
    axes[1].legend()
    
    # Precision-Recall kriva
    prec, rec, _ = precision_recall_curve(y_test, y_prob)
    axes[2].plot(rec, prec, color='steelblue', lw=2, label=f'PR-AUC = {pr_auc:.4f}')
    axes[2].set_xlabel('Recall')
    axes[2].set_ylabel('Precision')
    axes[2].set_title('Precision-Recall Kriva')
    axes[2].legend()
    
    plt.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
    plt.show()
    
    return {'title': title, 'roc_auc': roc_auc, 'pr_auc': pr_auc, 'f1': f1}


def find_best_threshold(model, X_val, y_val):
    """Pronalazi optimalni threshold po F1 na validation setu."""
    y_prob = model.predict(X_val, verbose=0).ravel()
    thresholds = np.linspace(0.1, 0.9, 81)
    f1s = [f1_score(y_val, (y_prob >= t).astype(int)) for t in thresholds]
    best_t = thresholds[np.argmax(f1s)]
    print(f'Optimalni threshold (F1): {best_t:.2f} → F1 = {max(f1s):.4f}')
    return best_t


print('✅ Helper funkcije definisane')

## 3. Definicija Baseline MLP arhitekture

In [ ]:
def build_baseline_mlp(input_dim, hidden_units=[64, 32, 16], dropout_rate=0.3, learning_rate=1e-3):
    """
    Baseline MLP za detekciju prevara.
    Arhitektura: Input -> Dense(64, ReLU) -> Dropout -> Dense(32, ReLU) -> Dropout -> Dense(16, ReLU) -> Output(sigmoid)
    """
    model = keras.Sequential(name='Baseline_MLP')
    model.add(layers.Input(shape=(input_dim,)))
    
    for units in hidden_units:
        model.add(layers.Dense(units, activation='relu',
                               kernel_initializer='he_normal',
                               kernel_regularizer=keras.regularizers.l2(1e-4)))
        model.add(layers.Dropout(dropout_rate))
    
    model.add(layers.Dense(1, activation='sigmoid'))
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=[
            keras.metrics.AUC(curve='PR', name='auc'),  # PR-AUC — glavna metrika
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='recall')
        ]
    )
    return model

# Pregled arhitekture
demo_model = build_baseline_mlp(INPUT_DIM)
demo_model.summary()

## 4. Callbacks

In [ ]:
def get_callbacks(model_name, patience=10):
    return [
        callbacks.EarlyStopping(
            monitor='val_auc', mode='max',
            patience=patience, restore_best_weights=True, verbose=1
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_auc', mode='max',
            factor=0.5, patience=5, min_lr=1e-6, verbose=1
        ),
        callbacks.ModelCheckpoint(
            filepath=f'{MODELS}{model_name}.keras',
            monitor='val_auc', mode='max',
            save_best_only=True, verbose=0
        )
    ]

## 5. Eksperiment A — Class Weight (bez resamplinga)

In [ ]:
print('Treniram: MLP + class_weight pristup')
print(f'Class weights: {class_weight}')

model_cw = build_baseline_mlp(INPUT_DIM)

history_cw = model_cw.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=256,
    class_weight=class_weight,
    callbacks=get_callbacks('mlp_classweight'),
    verbose=1
)

plot_history(history_cw, 'MLP + Class Weight', f'{RESULTS}03_mlp_cw_history.png')

## 6. Eksperiment B — SMOTE resampling

In [ ]:
print('Treniram: MLP + SMOTE')

model_smote = build_baseline_mlp(INPUT_DIM)

history_smote = model_smote.fit(
    X_train_smote, y_train_smote,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=256,
    callbacks=get_callbacks('mlp_smote'),
    verbose=1
)

plot_history(history_smote, 'MLP + SMOTE', f'{RESULTS}03_mlp_smote_history.png')

## 7. Eksperiment C — SMOTE + Undersampling

In [ ]:
print('Treniram: MLP + SMOTE + Undersampling')

model_comb = build_baseline_mlp(INPUT_DIM)

history_comb = model_comb.fit(
    X_train_comb, y_train_comb,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=256,
    callbacks=get_callbacks('mlp_combined'),
    verbose=1
)

plot_history(history_comb, 'MLP + SMOTE+Under', f'{RESULTS}03_mlp_comb_history.png')

## 8. Evaluacija i poređenje na Test setu

In [ ]:
results = []

for model, name, save in [
    (model_cw,    'MLP + Class Weight',    f'{RESULTS}03_eval_cw.png'),
    (model_smote, 'MLP + SMOTE',           f'{RESULTS}03_eval_smote.png'),
    (model_comb,  'MLP + SMOTE+Under',     f'{RESULTS}03_eval_comb.png'),
]:
    best_t = find_best_threshold(model, X_val, y_val)
    res = evaluate_model(model, X_test, y_test, threshold=best_t, title=name, save_path=save)
    res['threshold'] = best_t
    results.append(res)

In [ ]:
# Sumarno poređenje
results_df = pd.DataFrame(results).set_index('title')
print('\n=== SUMARNO POREĐENJE BASELINE MODELA ===')
display(results_df[['roc_auc', 'pr_auc', 'f1', 'threshold']]
        .style.highlight_max(color='#90EE90', subset=['roc_auc', 'pr_auc', 'f1']))

# Bar plot poređenje
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(results_df))
w = 0.25
ax.bar(x - w, results_df['roc_auc'], w, label='ROC-AUC', color='steelblue')
ax.bar(x,     results_df['pr_auc'],  w, label='PR-AUC',  color='coral')
ax.bar(x + w, results_df['f1'],      w, label='F1',       color='seagreen')
ax.set_xticks(x)
ax.set_xticklabels(results_df.index, rotation=10)
ax.set_ylim(0.7, 1.0)
ax.legend()
ax.set_title('Poređenje Baseline MLP Modela', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS}03_comparison.png', bbox_inches='tight')
plt.show()

# Čuvamo rezultate
results_df.to_csv(f'{RESULTS}03_baseline_results.csv')
print('\n→ Sledeći notebook: 04_advanced_nn.ipynb')